# Data exported new kicker measurement

In [1]:
import databroker
import datetime
import tqdm
import xarray as xr
import numpy as np

ModuleNotFoundError: No module named 'databroker'

## Data broker access 

In [2]:
from databroker import Broker, catalog

In [3]:
db = catalog['datascc']

In [4]:
db_old = Broker.named('datascc')

In [5]:
## Test runs: failed as bugs were still in the software

no beam position data ...

In [6]:
# uid = '7ac9b5b3-ef6d-495c-9126-82d0496f8ffd'
# uid = 'ed32186a-8eaf-4250-8d46-a5ece2856880'

reordered:

* first ring
* then track_one_turn

In [7]:
if False:
    uid = 'c68e5890-dcee-4e13-ba88-a45895d40058'
    uid = 'f207092c-20db-44a3-ade6-d99a71e6518c'
    uid = '8f911734-4aaf-4598-b37a-5dc5afcc8074'
    uid = '09237649-5f86-4a7f-9c7d-33b9fcb788fb'

    uid = 'a227fd11-c92d-42dd-9bae-c170d89592ba'



In [8]:
# Test run with 4 rays 

uid = '514c0b21'

In [9]:
# Run with 41 rays 
uid = '6d0c5144-3da0-44c5-a784-55ef23d6350e'

# Runs above 41'ths ray
uid = '6ccb6373-dd6f-4808-97d8-e4a1be69e041'


# These runs are broken as the y value contained the x values too


In [10]:
# Hopefully fixed data now with all runs insed
# but analysis still x and y are the same
uid = '7c52be40-3874-4f58-9090-c03ff2a1385c'

In [11]:
# Only three rays
# lifetable data indicate different maxima for x and y data

uid = 'b132f008-aa39-492c-b2bb-e77c530323ef'

## At this point not buggy data any more

But kicker is at lattice position 0 ... thus injection kicker not diagnostic kicker 

But mechanical aperture was not working yet 

In [12]:
# hopefully correct x and y data this time 
uid = 'd913ea0b-60af-4ce1-a299-e79076f4c92f'

save_name = 'kicker_turn_by_turn_all_rays.nc'

In [32]:
uid = '7d6b6c4d'
save_name = 'kicker_turn_by_turn_test.nc'

In [52]:
# hopefully correct x and y data this time 
#uid = 'd913ea0b-60af-4ce1-a299-e79076f4c92f'
uid = '951bf083-3941-4aae-ba0e-739d29a62ddf'


save_name = 'kicker_turn_by_turn_all_rays.nc'

In [72]:
# connected to the real machine  
#uid = 'd913ea0b-60af-4ce1-a299-e79076f4c92f'
uid = '27debbf6-5efe-451d-aebf-70196f09846b'

save_name = 'kicker_turn_by_turn_all_rays_connected.nc'

In [73]:
## Processing data 

In [74]:
header = db_old[uid]

In [75]:
run = db[uid]

In [76]:
run.metadata['start']['uid']

'27debbf6-5efe-451d-aebf-70196f09846b'

In [77]:
primary = run.primary()

In [78]:
desc, =  header['descriptors']

In [79]:
d = desc['configuration']['kd']['data']
bm_names = list(d['kd_bm_names'])
bm_ds = d['kd_bm_ds']

In [80]:
bm_names_cleared = bm_names.copy()
doublets = [name for name in bm_names if bm_names.count(name)> 1]

for cnt, name in enumerate(doublets):
    idx = bm_names.index(name)
    t_name = bm_names[idx] + '_{:d}'.format(cnt)
    bm_names_cleared[idx] = t_name 

In [81]:
start = datetime.datetime.now()
dask_ = primary.to_dask()
stop =  datetime.datetime.now()

In [82]:
266240 / 2048
262144 / 2048

128.0

In [83]:
replace_dims = {name : 'name' for name, dim in dask_.dims.items() if dim == len(bm_names) }
replace_dims

{'dim_0': 'name',
 'dim_1': 'name',
 'dim_2': 'name',
 'dim_3': 'name',
 'dim_4': 'name',
 'dim_5': 'name',
 'dim_6': 'name',
 'dim_7': 'name',
 'dim_8': 'name',
 'dim_9': 'name'}

In [84]:
dask = dask_.rename(replace_dims).assign_coords(name=bm_names_cleared)

In [85]:
start = datetime.datetime.now()
for varname in tqdm.tqdm(dask.variables, total=len(dask.variables)):
    var = getattr(dask, varname)
    var.load()
    
end = datetime.datetime.now()

100%|██████████| 65/65 [02:19<00:00,  2.15s/it]


In [86]:
dask

<xarray.Dataset>
Dimensions:                         (dim_10: 266240, dim_11: 266240, name: 1372, time: 6305)
Coordinates:
  * time                            (time) float64 1.621e+09 ... 1.621e+09
  * name                            (name) <U14 'begin' ... 'dg9l1d1r'
Dimensions without coordinates: dim_10, dim_11
Data variables: (12/63)
    kd_kk_x_angle                   (time) float64 0.0 0.01 0.02 ... 0.55 0.56
    kd_kk_x_rangle                  (time) float64 0.0 1e-05 ... 0.00055 0.00056
    kd_kk_x_setv                    (time) float64 0.0 1e-05 ... 0.00055 0.00056
    kd_kk_y_angle                   (time) float64 0.0 0.0 0.0 ... 0.82 0.82
    kd_kk_y_rangle                  (time) float64 0.0 0.0 ... 0.00082 0.00082
    kd_kk_y_setv                    (time) float64 0.0 0.0 ... 0.00082 0.00082
    ...                              ...
    kd_td_reset_flags               (time) float64 1.0 1.0 1.0 ... 1.0 1.0 1.0
    kd_td_ap_setpoint               (time) float64 1.0 1.0 1.0 ... 1.0 1.0 1.0
    kd_td_ap_readback               (time) float64 1.0 1.0 1.0 ... 1.0 1.0 1.0
    kd_td_ap_x_all                  (time) float64 0.05 0.05 0.05 ... 0.05 0.05
    kd_td_ap_y_all                  (time) float64 0.03 0.03 0.03 ... 0.03 0.03
    kd_ray_setpoint                 (time) float64 0.0 0.0 0.0 ... 90.0 90.0

In [92]:
zpos = xr.DataArray(name=bm_ds, data = bm_ds, dims = ['name'], coords=[bm_names_cleared])

In [93]:
dask_w_zpos = dask.merge(dict(zpos=zpos))

In [94]:
dask_w_zpos.to_netcdf(save_name)

In [95]:
dask.kd_bm_orbit_x_max

<xarray.DataArray 'kd_bm_orbit_x_max' (time: 6305)>
array([nan, nan, nan, ..., nan, nan, nan])
Coordinates:
  * time     (time) float64 1.621e+09 1.621e+09 ... 1.621e+09 1.621e+09
Attributes:
    object:   kd

In [96]:
save_name

'kicker_turn_by_turn_all_rays_connected.nc'